# Тетрадь 1.0: Кредитная политика своими руками

Это первая содержательная тетрадь проекта. Мы строим мультиагентный оркестратор кредитного андеррайтинга — сервис, который принимает заявку на кредит и решает, одобрить её или нет.

Но прежде чем писать агентов, надо ответить на простой вопрос: **а по каким правилам вообще решается судьба заявки**?

Этот шаг называется **«перевод бизнес-логики в требования»**. В реальной команде его делает системный аналитик вместе с бизнесом. В нашем MVP ты делаешь его сам, потому что иначе агентам не из чего будет выбирать.

**Чему ты научишься в этой тетради:**

- что такое кредитная политика и из чего она состоит;
- какие признаки заявки банк вообще смотрит;
- что такое DTI, PTI, скоринг и почему это главные метрики в кредитовании;
- как из правил на человеческом языке получается структура данных, которую сможет использовать агент;
- как хранить правила в JSON так, чтобы их было легко менять, не переписывая код.

**Что на выходе:**

Файл `data/policy.json` с 30–50 правилами, который дальше будет использоваться RAG-агентом (Policy Check Agent) во второй фазе проекта.

**Пункт чеклиста, который закрывается:**

- №7 (частично): переводить бизнес-логику и требования в работающий код;
- №8: общаться с бизнес-командами и разбираться в устройстве реальных процессов;
- №11 (частично): писать парсеры данных — правила собираются из открытых источников.

**Формат работы:**

Каждый раздел — короткая теория, пример и мини-задача. В конце — большой артефакт: `policy.json`. Не пропускай мини-задачи: без них не поймёшь, что происходит.

## 0. Как это связано с фазой 0

В фазе 0 мы построили фундамент: окружение через `uv`, структуру `src/underwriting/`, модуль `credit_math.py` с функциями `calc_dti`, `calc_payment`, `calc_loan_schedule`, модуль `schemas.py` со схемой `Application`, и генератор `generate.py`, который создаёт синтетические заявки.

**В этой тетради мы не будем дублировать то, что уже сделано.** Все расчёты (аннуитетный платёж, DTI, PTI, расписание платежей) уже реализованы в `src/underwriting/credit_math.py`. Мы просто импортируем их сюда и используем.

Это важное правило проекта: **код живёт в `src/`, тетрадь его импортирует, а не копирует**. Если ты найдёшь баг в `calc_dti`, ты исправишь его в одном месте, и это отразится везде — и в тетради, и в агенте.

Схема заявки из `schemas.py` тоже пригодится: правила политики ссылаются на те же поля (`monthly_income`, `loan_amount`, `employment_type`), которые там уже описаны. Не надо придумывать новые имена — используем существующие.

In [7]:
# Импортируем то, что уже сделано в фазе 0.
# Если этот блок падает — значит, окружение не настроено. Сначала пройди тетради 0.0–0.3.
from pathlib import Path
import json

from underwriting.credit_math import (
    calc_dti,
    calc_payment,
    calc_loan_schedule,
)
from underwriting.schemas import (
    Application,
    EmploymentType,
    LoanPurpose,
    CreditHistory,
    Region
)

print("Импорты из src/underwriting/ работают")

Импорты из src/underwriting/ работают


## 1. Что такое кредитная политика

**Кредитная политика банка** — это набор правил, по которым банк решает:

1. **давать ли кредит** конкретному клиенту;
2. **на каких условиях** — ставка, срок, лимит;
3. **что делать**, если клиент не проходит по части правил.

Политика — это не «одна формула». Это десятки-сотни правил, каждое из которых описывает одну сторону риска. Условно они делятся на четыре группы:

- **Скоринговая группа.** Внутренний балл банка: сколько клиент уже пользовался кредитами, были ли просрочки, есть ли залоги.
- **Доход и платёж.** Хватает ли у клиента денег, чтобы обслуживать новый кредит. Здесь живут DTI и PTI.
- **Профиль.** Возраст, занятость, регион, гражданство, стоп-факторы.
- **Особые случаи.** Ипотека, автокредит, рефинансирование — там свои правила.

Мы строим MVP по потребительскому кредитованию, поэтому будем работать с первыми тремя группами.

**Запомни главное:** правило политики — это всегда **предикат** (утверждение, которое либо верно, либо нет) плюс **действие**, если предикат верен. Например:

```
ЕСЛИ DTI > 50%
ТО отказ с причиной «предельная долговая нагрузка»
```

Мы будем хранить правила именно в таком виде.

## 2. Из чего состоит заявка на кредит

Чтобы правило могло сработать, у заявки должны быть **признаки**, на которые правило смотрит. Ещё раз: агент не может проверить правило, если в заявке нет нужного поля.

У нас эти признаки уже описаны в `src/underwriting/schemas.py` — в классе `Application`. Посмотрим, что там есть:

| Признак | Тип | Что означает |
|---|---|---|
| `application_id` | str | уникальный идентификатор заявки |
| `full_name` | str | ФИО клиента |
| `age` | int | возраст |
| `monthly_income` | float | средний месячный доход, руб |
| `monthly_debt` | float | уже существующие платежи, руб/мес |
| `loan_amount` | float | запрашиваемая сумма, руб |
| `loan_term_months` | int | срок кредита в месяцах |
| `loan_purpose` | LoanPurpose | цель кредита |
| `employment_type` | EmploymentType | тип занятости |
| `credit_history` | CreditHistory | качество кредитной истории |
| `applied_at` | date | дата подачи заявки |

Правила политики будут ссылаться на эти же поля. Именование должно совпадать один в один — иначе агент не найдёт поле в заявке.

Проверим это прямо в коде — создадим заявку и посмотрим на её поля.

In [8]:
# Быстрая проверка: заявка из схемы фазы 0.
from datetime import date

example = Application(
    application_id="test-001",
    full_name="Иванов Иван Иванович",
    age=35,
    region=Region.MOSCOW,
    monthly_income=80_000,
    monthly_debt=5_000,
    loan_amount=500_000,
    loan_term_months=36,
    loan_purpose=LoanPurpose.CONSUMER,
    employment_type=EmploymentType.EMPLOYED,
    credit_history=CreditHistory.CLEAN,
    applied_at=date.today(),
)

print(example.model_dump_json(indent=2))

{
  "application_id": "test-001",
  "full_name": "Иванов Иван Иванович",
  "age": 35,
  "region": "moscow",
  "monthly_income": 80000.0,
  "monthly_debt": 5000.0,
  "loan_amount": 500000.0,
  "loan_term_months": 36,
  "loan_purpose": "consumer",
  "employment_type": "employed",
  "credit_history": "clean",
  "applied_at": "2026-10-06"
}


## 3. Три ключевые метрики: DTI, PTI и ежемесячный платёж

Прежде чем писать правила, надо уметь считать три величины, на которых эти правила стоят.

**Ежемесячный платёж по кредиту** (аннуитетный) — уже реализован в `src/underwriting/credit_math.py` как `calc_payment(principal, annual_rate, months)`.

Формула, которую он реализует:

$$
M = P \cdot \frac{i \cdot (1+i)^n}{(1+i)^n - 1}
$$

где:
- `P` — сумма кредита;
- `i` — месячная ставка (`годовая / 12`);
- `n` — срок в месяцах.

**DTI (Debt-to-Income)** — долговая нагрузка:

$$
\text{DTI} = \frac{\text{ежемесячный платёж по новому кредиту} + \text{существующие платежи}}{\text{месячный доход}}
$$

**PTI (Payment-to-Income)** — нагрузка только по новому кредиту:

$$
\text{PTI} = \frac{\text{ежемесячный платёж по новому кредиту}}{\text{месячный доход}}
$$

Типичные пороги для российского потребительского кредитования (проверишь по открытым источникам в задаче 5.1):

- **DTI > 50%** — отказ;
- **DTI 40–50%** — снижение суммы или ручная проверка;
- **PTI > 40%** — почти всегда отказ.

### Мини-задача 3.1: используй существующий `calc_payment`

В фазе 0 ты уже написал `calc_payment(principal, annual_rate, months)` в `src/underwriting/credit_math.py`. Не пиши его заново. Просто импортируй и проверь на примере:

- сумма: 500 000 ₽
- ставка: 0.18 (то есть 18% в год — обрати внимание, `calc_payment` ожидает долю, а не проценты!)
- срок: 36 месяцев

Ожидаемый ответ — около **18 076 ₽** (может отличаться на пару рублей — это нормально, зависит от округления).

Если получилось — значит, код из фазы 0 работает правильно, и мы можем использовать его дальше.

In [9]:
payment = calc_payment(500_000, 0.18, 36)
print(f"Платёж: {payment:.2f} ₽")
assert abs(payment - 18_076) < 10, "Формула из фазы 0 дала неверный результат"
print("ОК: calc_payment из фазы 0 работает корректно")

Платёж: 18076.20 ₽
ОК: calc_payment из фазы 0 работает корректно


### Мини-задача 3.2: используй существующий `calc_dti`

В фазе 0 ты написал `calc_dti(monthly_debt, monthly_income)`, но он считает DTI **только по существующим долгам**. В кредитовании DTI включает ещё и новый платёж.

Здесь у тебя два варианта:

**Вариант A (быстрый):** посчитать DTI прямо в тетради:

```python
def full_dti(monthly_payment, existing_debt, monthly_income):
    return (monthly_payment + existing_debt) / monthly_income
```

**Вариант B (правильный):** добавить функцию `full_dti` и `pti` в `src/underwriting/credit_math.py`, покрыть тестами и импортировать в тетрадь.

Выбери вариант B — так код останется переиспользуемым, а тесты зафиксируют поведение. После этого проверь:

- доход 80 000 ₽, платёж 18 076 ₽, существующих долгов 5 000 ₽ →
  - PTI ≈ 0.226 (22.6%)
  - DTI ≈ 0.288 (28.8%)

In [10]:
# Сначала добавляем функции в src/underwriting/credit_math.py,
# потом импортируем. Пока проверим расчёт здесь, чтобы зафиксировать ожидания.

def pti(payment: float, monthly_income: float) -> float:
    return payment / monthly_income

def full_dti(payment: float, existing_debt: float, monthly_income: float) -> float:
    return (payment + existing_debt) / monthly_income

payment = calc_payment(500_000, 0.18, 36)
print(f"PTI: {pti(payment, 80_000):.3f}")
print(f"DTI: {full_dti(payment, 5_000, 80_000):.3f}")

PTI: 0.226
DTI: 0.288


## 4. Формат хранения правила

Мы будем хранить каждое правило как **JSON-объект** со следующими полями:

```json
{
  "id": "DTI_50",
  "category": "income",
  "description": "Если DTI больше 50%, выдать отказ.",
  "condition": {
    "metric": "dti",
    "operator": ">",
    "value": 0.5
  },
  "action": "reject",
  "reason": "Предельная долговая нагрузка (DTI > 50%)",
  "severity": "high",
  "source": "https://www.cbr.ru/..."
}
```

Разберём поля:

- `id` — уникальное имя правила, по нему на него ссылаются;
- `category` — группа (`income`, `profile`, `credit_history`, `special`);
- `description` — правило на человеческом языке. Это пойдёт в промпт LLM;
- `condition` — машиночитаемая часть:
  - `metric` — на какую величину смотрим (это либо признак заявки из `Application`, либо вычисляемая метрика типа `dti`);
  - `operator` — `>`, `>=`, `<`, `<=`, `==`, `in`, `not_in`;
  - `value` — с чем сравниваем;
- `action` — что делать: `approve`, `reject`, `review`, `lower_amount`, `info`;
- `reason` — почему так решили, эту причину увидит аналитик в финальном отчёте;
- `severity` — `high` / `medium` / `low`, влияет на приоритет при конфликте правил;
- `source` — ссылка на источник. Это **traceability** — прослеживаемость.

**Почему именно так?** Потому что агенту нужны **два представления**:

1. **Текст** (`description`) — чтобы LLM могла прочитать и интерпретировать в сложных случаях.
2. **Формальное условие** (`condition`) — чтобы код мог проверить быстро и без ошибок.

Это классический приём в AI-системах: правила, записанные в двух формах, — «текст для человека, код для машины».

### Мини-задача 4.1: напиши функцию, которая проверяет одно правило

Функция `check_rule(rule, application_metrics)` принимает:

- `rule` — словарь с правилом из JSON;
- `application_metrics` — словарь с вычисленными метриками заявки (там есть ключи `dti`, `pti`, `age`, `monthly_income` и т. п.).

Возвращает `True`, если условие **выполнено** (правило сработало), `False`, если нет.

Поддержи операторы `>`, `>=`, `<`, `<=`, `==`, `in`, `not_in`.

Проверь:

```python
rule = {"condition": {"metric": "dti", "operator": ">", "value": 0.5}}
check_rule(rule, {"dti": 0.62})  # должно быть True
check_rule(rule, {"dti": 0.31})  # должно быть False
```

Подумай, что должно случиться, если в `metrics` нет нужного ключа. Правильно — выбросить понятную ошибку. Не «молча False».

In [11]:
def check_rule(rule: dict, metrics: dict) -> bool:
    """
    Проверяет одно правило политики против метрик заявки.
    Возвращает True, если условие правила выполнено.
    """
    cond = rule["condition"]
    metric = cond["metric"]
    op = cond["operator"]
    value = cond["value"]

    if metric not in metrics:
        raise ValueError(f"Метрика '{metric}' отсутствует в заявке")

    actual = metrics[metric]

    if op == ">":
        return actual > value
    if op == ">=":
        return actual >= value
    if op == "<":
        return actual < value
    if op == "<=":
        return actual <= value
    if op == "==":
        return actual == value
    if op == "in":
        return actual in value
    if op == "not_in":
        return actual not in value

    raise ValueError(f"Неизвестный оператор: {op}")


rule = {"condition": {"metric": "dti", "operator": ">", "value": 0.5}}
print(check_rule(rule, {"dti": 0.62}))   # True
print(check_rule(rule, {"dti": 0.31}))   # False

# Проверка на отсутствующую метрику
try:
    check_rule(rule, {"age": 35})
except ValueError as e:
    print(f"Ожидаемая ошибка: {e}")

True
False
Ожидаемая ошибка: Метрика 'dti' отсутствует в заявке


## 5. Исследование: собери политику из открытых источников

Теперь самое интересное. **Реальные банки не публикуют свою политику целиком.** Но по открытым источникам (сайты банков, статьи, аналитика ЦБ РФ) можно восстановить **типичные правила**, которые используют российские банки для потребительского кредитования.

Твоя задача — собрать из открытых источников **30–50 правил** и записать их в формате из раздела 4.

**Где искать:**

- сайт ЦБ РФ — там публикуются нормативы и рекомендации;
- официальные страницы Альфа-Банка, Сбера, ВТБ про потребительские кредиты — там указаны требования к клиенту;
- статьи аналитиков (`banki.ru`, `frankrg.com`, `raexpert.ru`) — там разбираются типичные скоринговые правила.

**Что искать в первую очередь:**

- требования к возрасту (минимум/максимум);
- требования к доходу (минимум в месяц);
- требования к занятости (стаж на последнем месте);
- пороги DTI и PTI;
- правила по кредитной истории (просрочки, количество активных кредитов);
- ограничения по сумме в зависимости от дохода;
- базовые стоп-факторы (нет гражданства РФ, младше 21 года, старше 70, безработный и т. д.).

**Важно:** не выдумывай правила «от себя». Каждое правило должно иметь источник. Даже если это статья на `banki.ru` — впиши ссылку в поле `source`. Это называется **traceability** (прослеживаемость), и в реальной работе это требование №1.

### Мини-задача 5.1: собери 30 правил в файл policy.json

Сформируй файл `data/policy.json` — список правил. Каждое правило — объект по схеме из раздела 4.

**Структура файла:**

```json
{
  "version": "1.0",
  "source": "собрано из открытых источников, октябрь 2026",
  "rules": [ ... список правил ... ]
}
```

Не просто список, а объект с метаданными — так удобнее версионировать политику. Когда выйдет 2.0, можно будет сравнить и понять, какие правила изменились.

Минимум — по 5 правил на каждую из четырёх категорий (`income`, `profile`, `credit_history`, `special`), итого минимум 20. Хорошая работа — 30–50 правил с реальными порогами и осмысленными причинами.

**Проверка:** после того, как ты соберёшь файл, открой его программно и убедись, что каждое правило содержит все обязательные поля.

In [17]:
import json

# Open and load JSON file into a dictionary
with open('data/policy.json', 'r') as file:
    policy = json.load(file)

# Now 'data' is a dictionary
#print(type(data), len(data), '\n', data[0])  # <class 'dict'>

### Мини-задача 5.2: валидация

Напиши функцию `validate_policy(policy)`, которая:

1. Проверяет, что `policy` — объект с полями `version`, `source`, `rules`.
2. Проверяет, что `rules` — непустой список.
3. Для каждого правила проверяет наличие обязательных полей: `id`, `category`, `description`, `condition`, `action`, `reason`, `severity`, `source`.
4. Проверяет, что `condition` содержит `metric`, `operator`, `value`.
5. Проверяет, что `operator` — один из допустимых (`>`, `>=`, `<`, `<=`, `==`, `in`, `not_in`).
6. Проверяет, что `action` — один из допустимых (`approve`, `reject`, `review`, `lower_amount`, `info`).
7. Проверяет, что `severity` — один из (`high`, `medium`, `low`).
8. Проверяет, что все `id` уникальны.

Возвращает список ошибок. Если он пуст — политика валидна.

Прогони её на своём `policy.json`. Если есть ошибки — исправь.

In [20]:
def validate_policy(policy, verbose=True):
    """
    Проверяет политику на корректность. Возвращает (ok, errors).
    ok     : True, если ошибок нет
    errors : список сообщений об ошибках
    """
    required_rule_fields = {"id", "category", "description",
                            "condition", "action", "reason",
                            "severity", "source"}
    required_condition_fields = {"metric", "operator", "value"}

    valid_operators = {">", ">=", "<", "<=", "==", "in", "not_in"}
    valid_actions = {"approve", "reject", "review", "lower_amount", "info"}
    valid_severities = {"high", "medium", "low"}
    valid_categories = {"income", "profile", "credit_history", "special"}

    errors = []
    seen_ids = set()

    if not isinstance(policy, list):
        return False, ["policy должен быть списком правил"]

    for i, rule in enumerate(policy):
        prefix = f"Правило #{i}"

        # 0. Вообще словарь ли это
        if not isinstance(rule, dict):
            errors.append(f"{prefix}: не словарь, а {type(rule).__name__}")
            continue

        rid = rule.get("id", "<без id>")
        prefix = f"Правило '{rid}'"

        # 1. Обязательные поля правила
        missing = required_rule_fields - set(rule.keys())
        if missing:
            errors.append(f"{prefix}: не хватает полей {sorted(missing)}")
            continue  # дальше проверять нечего

        # 2. Уникальность id
        if rid in seen_ids:
            errors.append(f"{prefix}: дубликат id")
        seen_ids.add(rid)

        # 3. category
        if rule["category"] not in valid_categories:
            errors.append(
                f"{prefix}: неизвестная category '{rule['category']}'. "
                f"Допустимо: {sorted(valid_categories)}"
            )

        # 4. action
        if rule["action"] not in valid_actions:
            errors.append(
                f"{prefix}: неизвестная action '{rule['action']}'. "
                f"Допустимо: {sorted(valid_actions)}"
            )

        # 5. severity
        if rule["severity"] not in valid_severities:
            errors.append(
                f"{prefix}: неизвестная severity '{rule['severity']}'. "
                f"Допустимо: {sorted(valid_severities)}"
            )

        # 6. description и reason — непустые строки
        for f in ("description", "reason"):
            if not isinstance(rule[f], str) or not rule[f].strip():
                errors.append(f"{prefix}: поле '{f}' пустое или не строка")

        # 7. condition
        cond = rule["condition"]
        if not isinstance(cond, dict):
            errors.append(f"{prefix}: condition должен быть словарём")
            continue

        missing_cond = required_condition_fields - set(cond.keys())
        if missing_cond:
            errors.append(
                f"{prefix}: в condition не хватает полей {sorted(missing_cond)}"
            )
            continue

        if cond["operator"] not in valid_operators:
            errors.append(
                f"{prefix}: неизвестный operator '{cond['operator']}'. "
                f"Допустимо: {sorted(valid_operators)}"
            )

        if not isinstance(cond["metric"], str) or not cond["metric"].strip():
            errors.append(f"{prefix}: metric должен быть непустой строкой")

        # value: для in/not_in — список, для остальных — число или строка
        val = cond["value"]
        if cond["operator"] in ("in", "not_in"):
            if not isinstance(val, (list, tuple, set)):
                errors.append(
                    f"{prefix}: для operator '{cond['operator']}' "
                    f"value должен быть списком, а не {type(val).__name__}"
                )
        else:
            if not isinstance(val, (int, float, str, bool)):
                errors.append(
                    f"{prefix}: value должен быть числом или строкой, "
                    f"а не {type(val).__name__}"
                )

    ok = len(errors) == 0
    if verbose:
        if ok:
            print(f"✅ policy валидна: {len(policy)} правил, ошибок нет")
        else:
            print(f"❌ найдено ошибок: {len(errors)}")
            for e in errors:
                print("   -", e)
    return ok, errors

print(validate_policy(policy))

✅ policy валидна: 50 правил, ошибок нет
(True, [])


## 6. Как эту политику будет использовать агент

Здесь мы немного забегаем вперёд, чтобы ты видел цель.

В фазе 2 у нас будет **Policy Check Agent**. Он работает так:

1. Берёт заявку и вычисляет её метрики (`dti`, `pti`, `age`, `monthly_income` и т. п.).
2. Для каждой метрики достаёт из `policy.json` все правила, которые на неё смотрят. Это делает **RAG-поиск** по описаниям правил и по метрикам.
3. Передаёт найденные правила и метрики в LLM.
4. LLM проверяет: какие правила сработали, какие нет, какие конфликтуют. Возвращает структурированный ответ.
5. Результат пишется в БД и идёт дальше — в Pricing Agent и Brief Agent.

**Почему RAG, а не просто «загрузить всё в промпт»?** Потому что правил может быть много, а контекст LLM ограничен. Мы отбираем только релевантные — по метрике (`dti`, `age` и т. д.). Это классический подход «retrieve → augment → generate».

**Твоя работа в этой тетради — шаг 1 и часть шага 2.** Всё, что идёт дальше, — в фазе 2.

Запомни: **если `policy.json` плохой, то и агент будет работать плохо**. Нет «умного LLM», который спасёт кривую политику. Поэтому сейчас ты делаешь самую важную часть проекта.

## 7. Артефакт этой тетради

В конце у тебя должно быть:

1. **`data/policy.json`** — файл с 30+ правилами (в правильной структуре с `version`, `source`, `rules`).
2. **Функция `validate_policy`** — она либо внутри этой тетради, либо в отдельном скрипте `scripts/validate_policy.py`.
3. **Функция `check_rule`** — пригодится дальше в агенте.
4. **Дополнительно (правильный шаг):** перенести `check_rule` и `validate_policy` в `src/underwriting/policy.py`, покрыть тестами в `tests/test_policy.py`. Так код станет переиспользуемым, а не только доступным в тетради.
5. **Список источников** — файл `data/policy_sources.md` со ссылками, откуда взято каждое правило. Это требование traceability.

**Что делаем дальше:** в тетради 1.1 ты возьмёшь эти правила и дополнишь их ещё несколькими, уже спарсив сайты автоматически через `requests` + `BeautifulSoup`. Это будет твоя первая практика с парсерами.

## Что закрыто по чеклисту вакансии

После прохождения этой тетради у тебя отработаны (полностью или частично):

- **№7**: перевод бизнес-логики в код — правила на человеческом языке превращены в JSON и в функции-предикаты.
- **№8**: разбираешься в реальных процессах — читал кредитную политику банков, разбирался в DTI/PTI, скоринге.
- **№11** (начало): данные собираются из открытых источников — пока руками, в 1.1 будем автоматизировать.

**Что я ожидаю от тебя после этой тетради:**

1. Файл `data/policy.json` — твой, а не скопированный.
2. Файл `data/policy_sources.md` — со ссылками.
3. Функции `check_rule` и `validate_policy` — перенесены в `src/underwriting/policy.py`, покрыты тестами.
4. Короткий ответ на вопрос: «Почему кредитная политика не может быть одной формулой, а обязана состоять из десятков правил?» Этот ответ потом пойдёт в документацию проекта.

Когда закончишь — покажи мне артефакты, и я дам следующую тетрадь (1.1 — парсинг открытых источников).